In [ ]:
!pip uninstall -y bitsandbytes
!pip install bitsandbytes>=0.46.1

Сохраним итоги прошлой версии промптинга, чтобы было с чем сравнивать

Baseline

             MAE   RMSE   MBE  Correlation    QWK
    TR       0.99  1.151 -0.05        0.227  0.028
    CC       1.11  1.317 -0.27        0.187  0.022
    LR       1.07  1.279  0.15        0.136  0.015
    GRA      1.03  1.243  0.15        0.146  0.015
    Overall  1.07  1.219 -0.07        0.165  0.019

Контрастный промптинг
    
              MAE   RMSE   MBE  Correlation QWK
    TR       1.28  1.670  0.70        0.158  0.103
    CC       1.22  1.587  0.46        0.269  0.246
    LR       1.36  1.766  0.88        0.243  0.177
    GRA      1.34  1.735  0.88        0.271  0.161
    Overall  1.29  1.663  0.67        0.215  0.140

In [ ]:
# !pip install -q transformers accelerate bitsandbytes pandas datasets torch -- расскомментировать если такой штуки еще не установлено

import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
print(f"Устройство: {torch.cuda.get_device_name(0)}")
print(f"Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU доступен: True
Устройство: Tesla T4
Память: 15.6 GB


In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

# Это наш выбранный датасет с каггла, в нем данные сразу разбиты на тест и трейн

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d xuanhuynh233/ielts-dataset
!unzip ielts-dataset.zip

csv_path = 'train_final.csv'
df = pd.read_csv(csv_path)

print(df.head())
print(f"Размер датасета: {len(df)} строк.")

# Удалим строки, где 'essay' или 'band' отсутствуют,
# или 'band' не является числом.

initial_len = len(df)
df = df.dropna(subset=['essay', 'band'])
df = df[pd.to_numeric(df['band'], errors='coerce').notna()]
df['band'] = df['band'].astype(float) # Убедимся, что band это float
print(f"\nУдалено {initial_len - len(df)} строк с отсутствующими или некорректными 'essay'/'band'.")
print(f"Осталось {len(df)} строк после очистки.")

Mounted at /content/drive
cp: cannot stat '/content/drive/MyDrive/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/xuanhuynh233/ielts-dataset
License(s): MIT
100% 12.3M/12.3M [00:00<00:00, 65.8MB/s]

Archive:  ielts-dataset.zip
  inflating: test_final.csv          
  inflating: train_final.csv         
                                      band  \
0  7.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
1  5.0\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
2  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
3  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
4    4\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   

                                              prompt  \
0  Interviews form the basic criteria for most la...   
1  Interviews form the basic selecting criteria f...   
2  Interview form the basic selection criteria fo...   
3  Interviews form the basic selection criteria f...   
4  Interviews from the basi

In [ ]:
df.head(4)

,band,prompt,essay,TR_Band,TR_Comment,CC_Band,CC_Comment,LR_Band,LR_Mistakes,LR_Corrections,LR_Comment,GRA_Band,GRA_Mistakes,GRA_Corrections,GRA_Comment,Overall_Band,General_Feedback,error
0,7.5,Interviews form the basic criteria for most la...,It is believed by some experts that the tradit...,8.0,The essay sufficiently addresses all parts of ...,8.0,The essay is logically structured and the info...,7.0,exams writing; his teamwork skill is measured;...,written exams; their teamwork skills are measu...,The essay demonstrates a sufficient range of v...,7.0,...his ability to do the work and their proble...,...their ability to do the work and their prob...,"A variety of complex structures is used, and t...",7.5,This is a strong response with a very clear st...,NaN
1,5.0,Interviews form the basic selecting criteria f...,Nowadays numerous huge firms allocate an inter...,5.0,The essay addresses the task only partially. I...,5.0,"The essay is presented with some organization,...",5.0,choosing criteria; up-to-the-minute; reap the ...,selection criterion/criteria; effective/reliab...,The writer attempts to use some less common vo...,5.0,having excellent interpersonal skills are; mak...,having excellent interpersonal skills is; effe...,The essay shows a limited range of sentence st...,5.0,The essay presents a clear position and follow...,NaN
2,5.5,Interview form the basic selection criteria fo...,The interview section is the most vital part o...,6.0,"You have addressed all parts of the question, ...",6.0,The essay has a clear structure with an introd...,5.0,Interview section; prospect for the specified ...,The interview stage; prospective candidate for...,You have used an adequate range of vocabulary ...,5.0,Interview form the basic selection criteria; w...,Interviews form the basic selection criteria; ...,You have attempted to use a mix of simple and ...,5.5,Your essay presents a clear opinion and is log...,NaN
3,5.5,Interviews form the basic selection criteria f...,It is argued that the best method to recruit e...,6.0,The essay addresses the prompt by discussing b...,5.5,"The essay is organized into paragraphs, and th...",5.0,acquired by the recruiter; choosing the best e...,possessed by the applicant; choosing the best ...,The range of vocabulary is limited but minimal...,5.0,who do not lacks in any single question; accor...,who do not lack an answer to any single questi...,The writer attempts to use a mix of simple and...,5.5,Your essay has a clear structure and you prese...,NaN


In [ ]:
testing_exaples = df.sample(n=50, random_state=42) # зафиусируем выборку для тестирования
testing_exaples

,band,prompt,essay,TR_Band,TR_Comment,CC_Band,CC_Comment,LR_Band,LR_Mistakes,LR_Corrections,LR_Comment,GRA_Band,GRA_Mistakes,GRA_Corrections,GRA_Comment,Overall_Band,General_Feedback,error
621,5.5,"Nowadays, celebrities are more famous for thei...",It is widely argued that the young are negativ...,5.5,"The essay addresses the prompt, presenting ide...",6.0,"The essay is organized into paragraphs, and th...",5.0,earn budgets; swapped the concept; patronized ...,earn money; promoted the idea; endorsed/regula...,The writer attempts to use a range of vocabula...,5.5,the young are negatively affected by the glamo...,young people are negatively affected by the gl...,A mix of simple and complex sentence structure...,5.5,Your essay has a clear structure and you attem...,NaN
9120,4.5,Some people think that museums should be enjoy...,"In this enthusiastic world, museums are being ...",5.0,The essay attempts to address both views and p...,5.0,There is a clear attempt at a standard essay s...,4.0,enthusiastic world; entertaining locality; ann...,modern world; place of entertainment; historic...,The range of vocabulary is very limited and ba...,4.0,museums are being a hot topic; opportunities f...,museums are a popular topic; opportunities for...,The essay uses only a very limited range of gr...,4.5,You have attempted to structure the essay logi...,NaN
9257,6.0,"In some countries, older people are choosing t...",It is not uncommon that many old people today ...,6.0,The essay addresses the prompt and presents a ...,5.0,"The essay is organized into paragraphs, and th...",6.0,can be able to access to; Compare to living at...,can access / are able to access; Compared to l...,The essay uses an adequate range of vocabulary...,6.0,can be able to access to various infrastructur...,can access various forms of infrastructure; ol...,A mix of simple and complex sentence structure...,6.0,This essay presents a clear opinion and suppor...,NaN
9254,7.5,"In many countries today, parents are able to c...",It is undeniable that parents can prefer to en...,7.5,The response addresses all parts of the task a...,7.5,The essay is logically organised with a clear ...,7.5,do better academic outcomes; adamant proof; as...,achieve better academic outcomes; compelling e...,"The writer demonstrates a wide lexical range, ...",7.0,countries that applied this kind of teaching a...,countries that have applied this kind of teach...,A good range of grammatical structures is used...,7.5,This is a strong essay that demonstrates a sop...,NaN
9030,7.0,Some people think that museums should be enjoy...,It is argued that some people believe that mus...,7.0,The essay effectively addresses all parts of t...,7.0,The essay is well-organized with a logical str...,7.0,the other part ensures that; compare them with...,others maintain that; compare them with modern...,The candidate demonstrates a sufficient range ...,7.0,"For instance ,usually there are many trips...;...","For instance, there are usually many trips...;...",The essay shows a good range of grammatical st...,7.0,This is a strong response that successfully ad...,NaN
5915,5.5,Machines are taking over more and more jobs pr...,"In various nations around the world, there has...",6.0,"The essay addresses all parts of the prompt, d...",6.0,The essay is well-organized with a clear intro...,5.0,handled human jobs; finish people's jobs; crea...,jobs performed by humans; do people's jobs; ca...,The writer uses a sufficient range of vocabula...,5.0,what humans usually can do; it usually needs m...,what humans can usually do; it usually needs m...,The writer attempts to use a mix of simple and...,5.5,"The essay has a strong and logical structure, ...",NaN
5957,4.5,In many places women are taking jobs which are...,"Currently, females are opting for professions ...",5.0,The response addresses both parts of the promp...,5.0,The essay has a clear paragraph structure with...,4.0,has increased in a drastic manner; in various ...,has increased d

## загрузка QWEN

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Конфигурация 4-bit квантизации - чтобы быстро скачать и чтобы быстро работало (для пробы пера)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# сразу все импорты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr
import re
from tqdm import tqdm
import torch

In [ ]:
# функция оценки эссе по любому промпту
def grade_ielts_essay(essay, promt_func, example_essays, model, tokenizer, max_tokens=600):


    prompt = promt_func(essay, example_essays)
    # задаем роль
    messages = [
        {"role": "system", "content": "You are an expert IELTS Writing examiner with years of experience."},
        {"role": "user", "content": prompt}
    ]


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.2,  # Очень низкая для стабильности, была 0.3, выдавала разницу в качестве в 1 единцу MAE
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    return response

In [ ]:
# получим все оценки из ответа модели, чтобы считать по ним метрики качества

def extract_ielts_scores(response):

    # Извлечем 5 оценок {'TR': float, 'CC': float, 'LR': float, 'GRA': float, 'Overall': float}

    scores = {}

    patterns = {
        'TR': r'TR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'CC': r'CC[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'LR': r'LR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'GRA': r'GRA[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'Overall': r'Overall[_\s]*Band:\s*(\d+(?:\.\d)?)'
    }

    for criterion, pattern in patterns.items():
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            score = float(match.group(1))
            score = round(score * 2) / 2
            scores[criterion] = min(max(score, 0), 9)
        else:
            scores[criterion] = None

    if scores['Overall'] is None and all(scores[k] is not None for k in ['TR', 'CC', 'LR', 'GRA']):
        avg = np.mean([scores['TR'], scores['CC'], scores['LR'], scores['GRA']])
        scores['Overall'] = round(avg * 2) / 2

    return scores

In [ ]:
# оцениваем качество модели

def calculate_ielts_metrics(actual, predicted):

    mask = ~(pd.isna(actual) | pd.isna(predicted))
    actual = np.array(actual)[mask]
    predicted = np.array(predicted)[mask]

    if len(actual) == 0:
        return None

    # MAE
    mae = mean_absolute_error(actual, predicted)

    # RMSE
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    # MBE чтобы понимать занижение/завышение оценок
    mbe = np.mean(predicted - actual)

    # Корреляция Спирмена
    correlation, _ = spearmanr(actual, predicted)

    # Точность
    acc_05 = np.mean(np.abs(predicted - actual) <= 0.5)
    acc_10 = np.mean(np.abs(predicted - actual) <= 1.0)

    # Процент точных совпадений (особенно важно для 0.5 шага)
    exact_match = np.mean(predicted == actual)

    # QWK
    from sklearn.metrics import cohen_kappa_score
    qwk = cohen_kappa_score(
        (actual * 2).astype(int),  # Переводим в целые (умножаем на 2)
        (predicted * 2).astype(int),
        weights='quadratic'
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'MBE': mbe,
        'Correlation': correlation,
        'QWK': qwk,
        'Accuracy_±0.5': acc_05,
        'Accuracy_±1.0': acc_10,
        'Exact_Match': exact_match,
        'n_samples': len(actual)
    }

def quality(test_df):
  print("метрики качества")

  criteria_mapping = {
    'TR': 'TR_Band',
    'CC': 'CC_Band',
    'LR': 'LR_Band',
    'GRA': 'GRA_Band',
    'Overall': 'Overall_Band'
  }

  all_metrics = {}

  for criterion, actual_col in criteria_mapping.items():

    print(f" {criterion} ({actual_col})")
    actual = test_df[actual_col]
    predicted = test_df[f'pred_{criterion}']

    metrics = calculate_ielts_metrics(actual, predicted)

    if metrics:
        all_metrics[criterion] = metrics

        print(f"MAE:              {metrics['MAE']:.3f} bands")
        print(f"RMSE:             {metrics['RMSE']:.3f} bands")
        print(f"MBE:              {metrics['MBE']:+.3f} bands {'(завышение)' if metrics['MBE'] > 0 else '(занижение)' if metrics['MBE'] < 0 else '(нет смещения)'}")
        print(f"Корреляция:       {metrics['Correlation']:.3f}")
        print(f"QWK:              {metrics['QWK']:.3f}")
        print(f"Точность ±0.5:    {metrics['Accuracy_±0.5']:.1%}")
        print(f"Точность ±1.0:    {metrics['Accuracy_±1.0']:.1%}")
        print(f"Точное совпадение: {metrics['Exact_Match']:.1%}")
        print(f"Обработано:       {metrics['n_samples']} эссе")
    else:
        print(" Недостаточно данных для расчета метрик")
  return all_metrics


In [ ]:
def stats(all_metrics):
  metrics_df = pd.DataFrame(all_metrics).T
  metrics_df = metrics_df.round(3)
  print(metrics_df[['MAE', 'RMSE', 'MBE', 'Correlation', 'QWK']])

In [ ]:
metrics = ['TR_Band', 'CC_Band', 'LR_Band', 'GRA_Band', 'Overall_Band']
selected_indices = []

# Для каждой метрики ищем минимум и максимум
for metric in metrics:
    for extreme in ['min', 'max']:
        if extreme == 'min':
            # Сортируем по возрастанию, берём первое уникальное эссе
            sorted_df = df.sort_values(by=[metric, 'Overall_Band'], ascending=True)
        else:
            # Для максимума — сортируем по убыванию
            sorted_df = df.sort_values(by=[metric, 'Overall_Band'], ascending=False)

        # Проходим по отсортированным строкам, пока не найдём неиспользованный индекс
        for idx in sorted_df.index:
            if idx not in selected_indices:
                selected_indices.append(idx)
                break

contrast_df = df.loc[selected_indices].copy()
contrast_df

,band,prompt,essay,TR_Band,TR_Comment,CC_Band,CC_Comment,LR_Band,LR_Mistakes,LR_Corrections,LR_Comment,GRA_Band,GRA_Mistakes,GRA_Corrections,GRA_Comment,Overall_Band,General_Feedback,error
1645,4.0,It is the responsibility of schools to teach c...,The line graph illustrates the Outokumpu share...,1.0,The response is completely off-topic. The prom...,5.0,The response presents information with some or...,5.0,elevation of the share price; gradually inclin...,increase in the share price; gradually increas...,The writer uses a limited range of vocabulary ...,5.0,from January 2006 and December 2010; there is ...,between January 2006 and December 2010; there ...,A limited range of sentence structures is used...,4.0,The fundamental issue with this response is th...,NaN
136,9.0,The increase in the production of consumer goo...,"These days, there is an ever-growing number of...",9.0,The response fully addresses all parts of the ...,9.0,The essay is exceptionally well-organized. Par...,9.0,product manufacture; in the cable of minimizin...,product manufacturing; capable of minimizing t...,The writer demonstrates masterful control of a...,9.0,Consumer goods are attributed to various facto...,The environmental damage from consumer goods c...,A wide range of complex grammatical structures...,9.0,This is an exemplary response that fulfills al...,NaN
1185,4.0,1.Some people believe that studying at univers...,1. What does an airline clerk say when he/she ...,2.0,The response is entirely unrelated to the prom...,3.0,The text lacks any essay structure. It is pres...,5.0,passengers’s tickets; purpose his/her visit,passengers' tickets; purpose of his/her visit,The vocabulary is limited to the single topic ...,5.0,"he/she want to carry; Now, step I walk / go th...","he/she wants to carry; Now, please step/walk/g...",The response uses a very limited range of gram...,4.0,The most critical issue with this response is ...,NaN
147,9.0,The increase in the production of consumer goo...,"It is a horrendous truth, that rapid growth in...",9.0,The response fully addresses all parts of the ...,9.0,"The essay is skillfully managed, with a clear ...",9.0,product manufacturing causing the issues; good...,the manufacturing of products causes these iss...,The writer demonstrates a wide vocabulary and ...,9.0,"It is a horrendous truth, that rapid growth......",It is a horrendous truth that rapid growth...;...,A wide range of grammatical structures is used...,9.0,This is a very strong response that successful...,NaN
4,4.0,Interviews from the basic selecting criteria f...,Nowadays many companies conduct interviews bef...,4.0,The response addresses the task only in a mini...,4.0,"The essay presents information and ideas, but ...",4.0,chosing; intetviuvs; campaigns; sending them; ...,choosing; interviews; companies; hiring them; ...,The writer uses only basic and repetitive voca...,4.0,Interviews from the basic selecting criteria; ...,Interviews form the basic selection criteria; ...,Only a very limited range of sentence structur...,4.0,"Your essay attempts to address the prompt, but...",NaN
395,9.0,TASK 2: Some people think the money spent on d...,Some individuals claim that the amount of mone...,9.0,The response fully and expertly addresses all ...,9.0,The essay demonstrates masterful control of co...,9.0,world folk; human beings' specimen; the new cr...,the world's population / people worldwide; the...,"The use of vocabulary is wide-ranging, sophist...",9.0,is public funds consuming that could be utilis...,consumes public funds that could be utilised; ...,The essay showcases a wide range of complex gr...,9.0,This is an outstanding essay that demonstrates...,NaN
6119,4.0,Nowadays more and more jobs and tasks are done...,There are many companies and factories to comp...,5.0,The response attempts to address the prompt by...,4.0,The essay shows a basic attempt at paragraphin...,4.0,"to complete manual task, which is traditionall...","to complete manual tasks, which were 

In [ ]:
def testing(df, prompt_func, examples, sample_size=50, tokens=600):
  test_df = testing_exaples
  predictions = {
    'TR': [],
    'CC': [],
    'LR': [],
    'GRA': [],
    'Overall': []
  }
  explanations = []

  print(f"Оценка {sample_size} IELTS эссе...")

  for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        # Генерация оценки моделью по промту
        response = grade_ielts_essay(
            row['essay'],
            prompt_func,
            examples,
            model,
            tokenizer,
            max_tokens=tokens
        )

        # Извлечение оценок из ответа модели
        scores = extract_ielts_scores(response)

        for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
            predictions[criterion].append(scores[criterion])

        explanations.append(response)

    except Exception as e:
        print(f"\n Ошибка на эссе {idx}: {e}")
        for criterion in predictions.keys():
            predictions[criterion].append(None)
        explanations.append("")

  # Добавляем результаты в датафрейм
  for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
    test_df[f'pred_{criterion}'] = predictions[criterion]

  test_df['explanation'] = explanations

  print("\n Оценка завершена!")
  return test_df

## Добавление примеров комментариев в промпт

In [ ]:
def create_ielts_contrast_prompt_with_ans(essay_to_grade, example_essays):

    prompt = """You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

IMPORTANT:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria
- Provide realistic scores based on IELTS standards

Here are graded examples of good and bad scored essays. Pay attention to details of each:

"""

    for i, example in enumerate(example_essays, 1):
        prompt += f"""Example {i}:
Essay: "{example['essay']}"
TR_Band: {example['TR_Band']}
CC_Band: {example['CC_Band']}
LR_Band: {example['LR_Band']}
GRA_Band: {example['GRA_Band']}
Overall_Band: {example['Overall_Band']}
TR_Comment: {example['TR_Comment']}
GRA_Comment: {example['GRA_Comment']}
LR_Comment: {example['LR_Comment']}
CC_Comment: {example['CC_Comment']}
---

"""

    prompt += f"""Now, grade this essay following IELTS standards:

Essay: "{essay_to_grade}"

Provide your evaluation in this EXACT format in the begginning of your answer:
TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X
Explanation: [Brief justification for each criterion]
"""

    return prompt

In [ ]:
examples = pd.concat([contrast_df[:2], df.sample(2, random_state=42).copy()])# .to_dict('records')
examples

,band,prompt,essay,TR_Band,TR_Comment,CC_Band,CC_Comment,LR_Band,LR_Mistakes,LR_Corrections,LR_Comment,GRA_Band,GRA_Mistakes,GRA_Corrections,GRA_Comment,Overall_Band,General_Feedback,error
1645,4.0,It is the responsibility of schools to teach c...,The line graph illustrates the Outokumpu share...,1.0,The response is completely off-topic. The prom...,5.0,The response presents information with some or...,5.0,elevation of the share price; gradually inclin...,increase in the share price; gradually increas...,The writer uses a limited range of vocabulary ...,5.0,from January 2006 and December 2010; there is ...,between January 2006 and December 2010; there ...,A limited range of sentence structures is used...,4.0,The fundamental issue with this response is th...,NaN
136,9.0,The increase in the production of consumer goo...,"These days, there is an ever-growing number of...",9.0,The response fully addresses all parts of the ...,9.0,The essay is exceptionally well-organized. Par...,9.0,product manufacture; in the cable of minimizin...,product manufacturing; capable of minimizing t...,The writer demonstrates masterful control of a...,9.0,Consumer goods are attributed to various facto...,The environmental damage from consumer goods c...,A wide range of complex grammatical structures...,9.0,This is an exemplary response that fulfills al...,NaN
621,5.5,"Nowadays, celebrities are more famous for thei...",It is widely argued that the young are negativ...,5.5,"The essay addresses the prompt, presenting ide...",6.0,"The essay is organized into paragraphs, and th...",5.0,earn budgets; swapped the concept; patronized ...,earn money; promoted the idea; endorsed/regula...,The writer attempts to use a range of vocabula...,5.5,the young are negatively affected by the glamo...,young people are negatively affected by the gl...,A mix of simple and complex sentence structure...,5.5,Your essay has a clear structure and you attem...,NaN
9120,4.5,Some people think that museums should be enjoy...,"In this enthusiastic world, museums are being ...",5.0,The essay attempts to address both views and p...,5.0,There is a clear attempt at a standard essay s...,4.0,enthusiastic world; entertaining locality; ann...,modern world; place of entertainment; historic...,The range of vocabulary is very limited and ba...,4.0,museums are being a hot topic; opportunities f...,museums are a popular topic; opportunities for...,The essay uses only a very limited range of gr...,4.5,You have attempted to structure the essay logi...,NaN


In [ ]:
# добавим пару рандомных примеров
examples = pd.concat([contrast_df[:2], df.sample(2, random_state=42).copy()]).to_dict('records')
test_df = testing(df, create_ielts_contrast_prompt_with_ans, examples, sample_size=50, tokens=2000)
test_df.head(3)

Оценка 50 IELTS эссе...


100%|██████████| 50/50 [1:06:57<00:00, 80.35s/it]


 Оценка завершена!


,band,prompt,essay,TR_Band,TR_Comment,CC_Band,CC_Comment,LR_Band,LR_Mistakes,LR_Corrections,...,GRA_Comment,Overall_Band,General_Feedback,error,pred_TR,pred_CC,pred_LR,pred_GRA,pred_Overall,explanation
621,5.5,"Nowadays, celebrities are more famous for thei...",It is widely argued that the young are negativ...,5.5,"The essay addresses the prompt, presenting ide...",6.0,"The essay is organized into paragraphs, and th...",5.0,earn budgets; swapped the concept; patronized ...,earn money; promoted the idea; endorsed/regula...,...,A mix of simple and complex sentence structure...,5.5,Your essay has a clear structure and you attem...,NaN,5.5,5.0,5.0,5.5,5.5,### Essay Evaluation\n\n#### TR_Band: 5.5\n**E...
9120,4.5,Some people think that museums should be enjoy...,"In this enthusiastic world, museums are being ...",5.0,The essay attempts to address both views and p...,5.0,There is a clear attempt at a standard essay s...,4.0,enthusiastic world; entertaining locality; ann...,modern world; place of entertainment; historic...,...,The essay uses only a very limited range of gr...,4.5,You have attempted to structure the essay logi...,NaN,NaN,NaN,NaN,NaN,NaN,### Evaluation:\n\n**TR_Band:** 5.0 \n**CC_Ba...
9257,6.0,"In some countries, older people are choosing t...",It is not uncommon that many old people today ...,6.0,The essay addresses the prompt and presents a ...,5.0,"The essay is organized into paragraphs, and th...",6.0,can be able to access to; Compare to living at...,can access / are able to access; Compared to l...,...,A mix of simple and complex sentence structure...,6.0,This essay presents a clear opinion and suppor...,NaN,5.0,5.0,4.0,4.0,4.5,### Evaluation:\n\n**TR_Band: 5.0**\n**CC_Band...


In [ ]:
all_metrics = quality(test_df)

метрики качества
 TR (TR_Band)
MAE:              1.163 bands
RMSE:             1.443 bands
MBE:              -0.581 bands (занижение)
Корреляция:       -0.162
QWK:              -0.090
Точность ±0.5:    30.2%
Точность ±1.0:    55.8%
Точное совпадение: 23.3%
Обработано:       43 эссе
 CC (CC_Band)
MAE:              1.407 bands
RMSE:             1.727 bands
MBE:              -0.942 bands (занижение)
Корреляция:       -0.090
QWK:              -0.023
Точность ±0.5:    23.3%
Точность ±1.0:    51.2%
Точное совпадение: 23.3%
Обработано:       43 эссе
 LR (LR_Band)
MAE:              1.500 bands
RMSE:             1.803 bands
MBE:              -0.756 bands (занижение)
Корреляция:       -0.167
QWK:              -0.120
Точность ±0.5:    14.0%
Точность ±1.0:    58.1%
Точное совпадение: 9.3%
Обработано:       43 эссе
 GRA (GRA_Band)
MAE:              1.442 bands
RMSE:             1.755 bands
MBE:              -0.674 bands (занижение)
Корреляция:       -0.240
QWK:              -0.185
Точность ±0.5:   

In [ ]:
stats(all_metrics)

           MAE   RMSE    MBE  Correlation    QWK
TR       1.163  1.443 -0.581       -0.162 -0.090
CC       1.407  1.727 -0.942       -0.090 -0.023
LR       1.500  1.803 -0.756       -0.167 -0.120
GRA      1.442  1.755 -0.674       -0.240 -0.185
Overall  1.430  1.683 -0.779       -0.207 -0.139


Кажется, что модель не чувствует разницу между оценками, попробуем передавать не рандомные примеры, а по примеру на каждую оценку. Так получается слишком длинный промпт, модель не ест.

Кажется, что нам именно важно научить правильным мыслям, попробуем передавать только комментарии экпертов и оценку, без текста эссе.


In [ ]:
def create_ielts_prompt_only_comms(essay_to_grade, example_essays):

    prompt = """You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

IMPORTANT:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria
- Provide realistic scores based on IELTS standards

Here are comments of experts of good and bad scored essays. Pay attention to details of each:

"""

    for i, example in enumerate(example_essays, 1):
        prompt += f"""Example {i}:
TR_Band: {example['TR_Band']}
CC_Band: {example['CC_Band']}
LR_Band: {example['LR_Band']}
GRA_Band: {example['GRA_Band']}
Overall_Band: {example['Overall_Band']}
TR_Comment: {example['TR_Comment']}
GRA_Comment: {example['GRA_Comment']}
LR_Comment: {example['LR_Comment']}
CC_Comment: {example['CC_Comment']}
---

"""

    prompt += f"""Now, grade this essay following IELTS standards:

Essay: "{essay_to_grade}"

Provide your evaluation in this EXACT format in the begginning of your answer:
TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X
Explanation: [Brief justification for each criterion]
"""

    return prompt

In [ ]:
import random
selected_indices = [
    random.choice(df[df['Overall_Band'] == score].index.tolist())
    for score in range(4, 10)
    if len(df[df['Overall_Band'] == score]) > 0
]

all_ex_df = df.loc[selected_indices].copy()

In [ ]:
all_ex_df.shape

(6, 18)

In [ ]:
examples = all_ex_df.to_dict('records')
test_df = testing(df, create_ielts_prompt_only_comms, examples, sample_size=50, tokens=2000)
test_df.head(3)

Оценка 50 IELTS эссе...


100%|██████████| 50/50 [45:21<00:00, 54.44s/it]


 Оценка завершена!


,band,prompt,essay,TR_Band,TR_Comment,CC_Band,CC_Comment,LR_Band,LR_Mistakes,LR_Corrections,...,GRA_Comment,Overall_Band,General_Feedback,error,pred_TR,pred_CC,pred_LR,pred_GRA,pred_Overall,explanation
621,5.5,"Nowadays, celebrities are more famous for thei...",It is widely argued that the young are negativ...,5.5,"The essay addresses the prompt, presenting ide...",6.0,"The essay is organized into paragraphs, and th...",5.0,earn budgets; swapped the concept; patronized ...,earn money; promoted the idea; endorsed/regula...,...,A mix of simple and complex sentence structure...,5.5,Your essay has a clear structure and you attem...,NaN,6.5,6.5,6.5,6.5,6.5,**TR_Band: 6.5 \nCC_Band: 6.5 \nLR_Band: 6.5...
9120,4.5,Some people think that museums should be enjoy...,"In this enthusiastic world, museums are being ...",5.0,The essay attempts to address both views and p...,5.0,There is a clear attempt at a standard essay s...,4.0,enthusiastic world; entertaining locality; ann...,modern world; place of entertainment; historic...,...,The essay uses only a very limited range of gr...,4.5,You have attempted to structure the essay logi...,NaN,6.0,6.0,6.0,6.0,6.0,**TR_Band: 6.0 \nCC_Band: 6.0 \nLR_Band: 6.0...
9257,6.0,"In some countries, older people are choosing t...",It is not uncommon that many old people today ...,6.0,The essay addresses the prompt and presents a ...,5.0,"The essay is organized into paragraphs, and th...",6.0,can be able to access to; Compare to living at...,can access / are able to access; Compared to l...,...,A mix of simple and complex sentence structure...,6.0,This essay presents a clear opinion and suppor...,NaN,6.0,6.0,6.0,6.0,6.0,**TR_Band: 6.0 \nCC_Band: 6.0 \nLR_Band: 6.0...


In [ ]:
all_metrics = quality(test_df)

метрики качества
 TR (TR_Band)
MAE:              1.090 bands
RMSE:             1.366 bands
MBE:              +0.430 bands (завышение)
Корреляция:       -0.028
QWK:              0.016
Точность ±0.5:    34.0%
Точность ±1.0:    64.0%
Точное совпадение: 16.0%
Обработано:       50 эссе
 CC (CC_Band)
MAE:              1.190 bands
RMSE:             1.440 bands
MBE:              +0.210 bands (завышение)
Корреляция:       -0.048
QWK:              0.000
Точность ±0.5:    30.0%
Точность ±1.0:    60.0%
Точное совпадение: 12.0%
Обработано:       50 эссе
 LR (LR_Band)
MAE:              1.310 bands
RMSE:             1.592 bands
MBE:              +0.630 bands (завышение)
Корреляция:       -0.113
QWK:              -0.060
Точность ±0.5:    32.0%
Точность ±1.0:    48.0%
Точное совпадение: 12.0%
Обработано:       50 эссе
 GRA (GRA_Band)
MAE:              1.270 bands
RMSE:             1.554 bands
MBE:              +0.630 bands (завышение)
Корреляция:       -0.099
QWK:              -0.054
Точность ±0.5:    

In [ ]:
stats(all_metrics)

          MAE   RMSE   MBE  Correlation    QWK
TR       1.09  1.366  0.43       -0.028  0.016
CC       1.19  1.440  0.21       -0.048  0.000
LR       1.31  1.592  0.63       -0.113 -0.060
GRA      1.27  1.554  0.63       -0.099 -0.054
Overall  1.19  1.454  0.41       -0.102 -0.044


Итоги 2 экспериментов:

Промпт с эссе и комментариями

               MAE   RMSE    MBE  Correlation    QWK
    TR       1.163  1.443 -0.581       -0.162 -0.090
    CC       1.407  1.727 -0.942       -0.090 -0.023
    LR       1.500  1.803 -0.756       -0.167 -0.120
    GRA      1.442  1.755 -0.674       -0.240 -0.185
    Overall  1.430  1.683 -0.779       -0.207 -0.139

Промпт только с комментариями

              MAE   RMSE   MBE  Correlation    QWK
    TR       1.09  1.366  0.43       -0.028  0.016
    CC       1.19  1.440  0.21       -0.048  0.000
    LR       1.31  1.592  0.63       -0.113 -0.060
    GRA      1.27  1.554  0.63       -0.099 -0.054
    Overall  1.19  1.454  0.41       -0.102 -0.044

 Отрицательная корреляция означает, что модель предсказывает оценку, которая противоположна реальной: если эссе на самом деле хорошее, модель предсказывает низкую оценку, и наоборот. Отрицательный QWK означает, что согласие модели с человеческими оценками хуже, чем случайное. Поэтому пока оба эскперимента признаем неудачными